In [18]:
import os

import mysql.connector as conn
import numpy as np
import pandas as pd; pd.set_option("display.float_format","{:,.2f}".format)
import sqlalchemy
from dotenv import load_dotenv
from sqlalchemy.sql import text

In [4]:
# Takes environment variables from .env.
load_dotenv()

# Credentials for local PostgreSQL database.
pg_host = "localhost"
pg_database = "paid_content"
pg_schema = "user_profile"
pg_table = "activity"
pg_user = "postgres"
pg_pass = os.environ["POSTGRES_PASSWORD"]
pg_port = 5432

# Credentials for internal Delfi database.
dlf_name_articles = "delfi_articles"
dlf_name_paywall = "paywall"
dlf_host = os.environ["DELFI_DB_HOST"]
dlf_user = os.environ["DELFI_DB_USER"]
dlf_pass = os.environ["DELFI_DB_PASSWORD"]
dlf_port = os.environ["DELFI_DB_PORT"]

# MySQL connector.
delfi_paywall_conn = conn.connect(
    host=dlf_host,
    port=dlf_port,
    user=dlf_user,
    password=dlf_pass,
    database=dlf_name_paywall
)

In [15]:
# Query all data about activity of paid content users from local PostgreSQL database.
postgresql_url = f"postgresql+psycopg2://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_database}"

engine = sqlalchemy.create_engine(postgresql_url)

activity_query = text("SELECT * FROM user_profile.activity WHERE user_delfi_id NOT IN (1415556, 963890, 930767)")

with engine.connect().execution_options(autocommit=True) as conn:
    activity_query_sql = conn.execute(activity_query)

initial_activity_data = pd.DataFrame(activity_query_sql.fetchall())

initial_activity_data["activity_date"] = initial_activity_data["timestamp"].dt.date
initial_activity_data["activity_year"] = initial_activity_data["timestamp"].dt.year
initial_activity_data["activity_month"] = initial_activity_data["timestamp"].dt.month

unique_users = initial_activity_data["user_delfi_id"].nunique()
num_row = len(initial_activity_data)
print(f"Initial data consists of {num_row} rows and {unique_users} unique users.")

Initial data consists of 432696 rows and 14065 unique users.


Here and in the subsequent calculations of average values I use median because distribution of all data is more or less skewed to the rigth. 

In [19]:
rfv_df = initial_activity_data.groupby("user_delfi_id").agg(
    frequency=("activity_date", lambda x: x.nunique()),
    volume=("article_id", lambda x: x.nunique()),
    last_visit=("activity_date", "max"),
    avg_active_time=("active_time", "median"),
    avg_scroll_depth=("scroll_depth", "median")
).reset_index()

# The latest date is literally the lastest date of data and is used to calculate recency.
latest_date = "2023-03-14"
anchor_date = pd.to_datetime(latest_date).date()
rfv_df["recency"] = (anchor_date - rfv_df["last_visit"]).dt.days

# Data with final RFV score.
rfv_df["rfv_score"] = (rfv_df["frequency"] * np.sqrt((rfv_df["volume"]))) / (rfv_df["recency"] + 1)
rfv_df["user_delfi_id"] = rfv_df["user_delfi_id"].astype("string")
rfv_df.head()

,user_delfi_id,frequency,volume,last_visit,avg_active_time,avg_scroll_depth,recency,rfv_score
0,1735,3,3,2023-03-01,6.50,39.00,13,0.37
1,2140,43,74,2023-03-11,250.50,60.50,3,92.47
2,2320,36,50,2023-03-14,91.00,48.00,0,254.56
3,2872,8,10,2023-03-14,47.50,98.00,0,25.30
4,3069,34,70,2023-03-14,63.00,50.50,0,284.46


I tried to select just those users who were active during the whole period: theirs subscriptions valid_till date was greater than the upper bound of period and first purchase date was older than the lower bound of period. With the date of first purchase I eliminate those users who bought theirs first subscriptions anytime during the period. Hence in theory all users have the equal possibilities to visit website the same number of times and read the same amount of articles.

When time period is increasing also increasing the probability that some users would churn and buy subscription again after some break (aka returned user). Due to constraints of our database I cannot eliminate such behavior but in our case number of such users is quite small.

In [ ]:
# Users who were active during whole period.
filtered_users_query = """
    SELECT P1.user_id, P1.plan, P1.renewal
    FROM (SELECT T.user_id, T.completion_timestamp, S.plan, US.renewal,
        ROW_NUMBER() OVER (PARTITION BY T.user_id ORDER BY T.completion_timestamp) AS row_num
        FROM paywall.transactions AS T
		LEFT JOIN paywall.user_subscriptions AS US
		ON T.user_id = US.user_id
		LEFT JOIN paywall.subscriptions AS S
		ON US.subscription_id = S.id
		WHERE US.service_id IN (1, 3, 4, 9, 10)
		AND S.included_in_count = 1
		AND T.completion_timestamp IS NOT NULL
		AND S.sf_category = "Paid subscriptions"
		AND US.valid_till > "2023-03-13 23:59:59") AS P1
    WHERE P1.row_num = 1
    AND P1.completion_timestamp < "2023-01-01 00:00:00"
    """

filtered_users = pd.read_sql(filtered_users_query, delfi_paywall_conn)
filtered_users = filtered_users.astype("string")
filtered_users.head()